# Approach 3 — All Recommended Experiments
One notebook for D3, R1, F2, R5, CSP (R4), Troika (R3) and SigLIP2 F3/F4. Read README first. Defaults are audit-only; no weights download and no training starts just by opening this notebook.

## Setup
Select the **COS30082 Experiment Suite** kernel. Set SELECTED to the IDs you want. Keep MODE="smoke" until each selected method passes on the school GPU. Set RUN=True only after inspecting the configuration. Starting a fresh session creates fresh output directories.

In [ ]:
from pathlib import Path
import sys, json, torch
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common/dataset.py").is_file())
sys.path.insert(0, str(ROOT))
from approach3_experiment_suite.suite import Suite
CONFIG = json.loads((ROOT / "approach3_experiment_suite/config.json").read_text())
SELECTED = ["D3"]  # Recommended order: D3,R1,F2,R5,R4,R3,F3,F4
MODE = "smoke"  # Change to full only after smoke passes
RUN = False
ALLOW_DOWNLOAD = False
E1_PATH = "checkpoints/dino_fusion_inputs/E1_wise.pt"
CONFIG.update(selected=SELECTED, e1_path=E1_PATH)
print(torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## Data
Official splits and333 candidate rows are unchanged. New trainable models use6119 images and1104 training-only compositional validation images. D3 preserves frozen E1 and all7223 original prototype images. Truly unseen crops/diseases are N/A under this dataset.

In [ ]:
suite = Suite(CONFIG, mode=MODE, allow_download=ALLOW_DOWNLOAD)
print("New session:", suite.path)
print(json.dumps(suite.audit["split"], indent=2))

## Model
D3 uses your exact E1 checkpoint and DINOv2. R1/R5/CSP/Troika use pristine CLIP B/16; F2/F3/F4 use SigLIP2. CSP/Troika are documented DPD/OpenCLIP adaptations. Models load only in the Evaluation execution cell, one experiment at a time.

In [ ]:
display(suite.catalog())
print("Selected:", SELECTED)
print("Downloads permitted:", ALLOW_DOWNLOAD)

## Training
D3/R1/F2/R5 have no gradient training. R4 learns composable vocabulary, R3 learns prompts/adapters/CMT, F3/F4 update only final SigLIP2 visual blocks. Checkpoints select on compositional validation; tests never select hyperparameters. The runner handles actual training before test evaluation. No final7223-image retraining is performed.

In [ ]:
print(json.dumps(suite.config, indent=2))
print("RUN:", RUN, "MODE:", MODE)
print("Full training defaults: 10 epochs, batch8, accumulation4. Smoke overrides are explicitly recorded.")

## Evaluation
Run only SELECTED methods. Missing weights stop unless you explicitly permit downloads. D3 full mode must reproduce original E1 clean/robustness results before fusion. R1 cannot tune E1 on the additional holdout because E1 has already trained on it. R5 has six sourced descriptions and explicit name-only fallback for the other labels.

In [ ]:
if RUN:
    for method in SELECTED:
        print("Running", method)
        print(suite.run(method))
else:
    print("Audit only: no model download, neural training or test evaluation.")

## Save
All outputs use unique new session/method directories. all_comparisons.csv is the combined table; session_index.json points to each detailed result. Copy the full new session folder off the school computer. Smoke metrics are mechanics checks, not formal research results.

In [ ]:
saved = suite.save()
print("Saved session:", saved)
comparison = saved / "all_comparisons.csv"
if comparison.exists():
    import pandas as pd
    display(pd.read_csv(comparison))
else:
    print("No model comparisons in audit-only mode.")